# Typical LP Problems: Be the Decision Maker
**IN2040 · Optimization of Organizational Processes · Tecnológico de Monterrey** (Prof. Vincent Boyer)

*Session 2 · explore during class, about 5 minutes per problem.*

> **How to use this notebook.** Run the **Setup** cell first. Then, for each problem:
> 1. read the **problem**: its data (prices, capacities, demands…) are **fixed**: they describe the situation;
> 2. play with the **explorer**: the sliders are the **decision variables**. Move them and watch the objective
>    and every constraint react (✔ satisfied, ✘ violated). Try to find the best feasible plan, then press
>    **Load the optimal plan** to compare with the solver;
> 3. read the **formulation**, then run the **PuLP** cell: the same model solved in Python.

For each problem, ask yourself:
1. **Which constraints stop you** when you try to improve the objective?
2. Which constraints are **binding** at the optimal plan (used exactly at their limit)?
3. How far was **your** plan from the optimum, and why?

In [ ]:
# @title ⚙️ Setup: run this cell first (no need to read the code)
import importlib, subprocess, sys
import importlib.metadata
try:                                   # PuLP 4 + its CBC solver (pulp[cbc]); replaces an older PuLP 3
    _old = int(importlib.metadata.version("pulp").split(".")[0]) < 4 or importlib.util.find_spec("cbcbox") is None
except importlib.metadata.PackageNotFoundError:
    _old = True
if _old:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[cbc]>=4"], check=True)
import base64, io, json, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from scipy.optimize import linprog

PARTS = []      # HTML pieces of the current view, written into one HTML widget (no Output widget: VS Code safe)


def optimum(c, A_ub=None, b_ub=None, A_eq=None, b_eq=None, bounds=None, maximize=False):
    """Optimal solution of the LP (HiGHS), used for the "load the optimal plan" button."""
    c = np.asarray(c, float)
    r = linprog(-c if maximize else c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    return r.x + 0.0, (-r.fun if maximize else r.fun)


def banner(ok, text, top=False):
    color, bg = ("#067429", "#eaf9ef") if ok else ("#bd0a0a", "#fff0f0")
    (lambda h: PARTS.insert(0, h) if top else PARTS.append(h))(f'<div style="border:2px solid {color};background:{bg};color:{color};padding:8px 12px;'
                 f'border-radius:8px;font-size:15px;margin:6px 0">{text}</div>')


def table(df):
    PARTS.append(df.to_html(index=False, border=1, escape=False,
                            float_format=lambda v: f"{0.0 if abs(v) < 5e-3 else v:,.2f}")
                 .replace('<table', '<table style="border-collapse:collapse;margin:0 0 8px 0;font-size:13px"', 1))


def show(fig):
    fig.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    PARTS.append(f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"/>')


def constraints(rows):
    """rows: (constraint, value of the left-hand side, sense, right-hand side). Shows a ✔/✘ table, returns the violated ones."""
    tol = 1e-4
    ok = [(v <= r + tol) if s == "≤" else (v >= r - tol) if s == "≥" else abs(v - r) <= tol for _, v, s, r in rows]
    table(pd.DataFrame({"Constraint": [n for n, *_ in rows], "Your plan": [v for _, v, _, _ in rows],
                        "": [s for _, _, s, _ in rows], "Limit (data)": [r for *_, r in rows],
                        "OK?": ["✔" if o else '<b style="color:#bd0a0a">✘</b>' for o in ok]}))
    return [n for (n, *_), o in zip(rows, ok) if not o]


def verdict(violated, z, z_best, what, maximize=False, unit="$"):
    """Banner: feasible or not, value of the objective, comparison with the optimum."""
    money = lambda v: f"{unit}{v:,.2f}" if unit == "$" else f"{v:,.2f} {unit}"
    if violated:
        banner(False, f"<b>INFEASIBLE plan</b>: it violates {', '.join(violated)}.<br>{what} = {money(z)}, but this "
                      f"plan is not allowed. Change the decisions until every constraint shows ✔.", top=True)
        return
    gap = (z_best - z) if maximize else (z - z_best)
    tail = ("<b>This is the optimal plan.</b>" if abs(gap) < 1e-3 * max(1, abs(z_best)) else
            f"The best feasible plan reaches {money(z_best)}: you are {money(abs(gap))} "
            f"{'below' if maximize else 'above'} it. Can you do better?")
    banner(True, f"<b>Feasible plan</b>: {what} = {money(z)}.<br>{tail}", top=True)


def slider(desc, value, lo, hi, step):
    return W.FloatSlider(value=value, min=lo, max=hi, step=step, description=desc, continuous_update=False,
                         style={"description_width": "150px"}, layout=W.Layout(width="420px"))


def explorer(groups, view, best, chart_beside_controls=False):
    """groups: {"Column title": {name: widget}}; best: {name: optimal value}. Redraws on every change.
    chart_beside_controls: the chart goes next to the controls (e.g. a map to look at while ticking roads)."""
    controls = {k: w for g in groups.values() for k, w in g.items()}
    cols = [W.VBox([W.HTML(f"<b>{title}</b>")] + list(g.values())) for title, g in groups.items()]
    out, chart = W.HTML(), W.HTML()

    def refresh(_=None):
        PARTS.clear()
        try:
            view(**{k: w.value for k, w in controls.items()})
        except Exception:
            plt.close("all")
            PARTS.append(f"<pre style='color:#bd0a0a'>{traceback.format_exc()}</pre>")
        # layout: verdict banner on top, then tables (left) and chart (right) side by side, so it fits one screen
        banners = [h for h in PARTS if h.startswith("<div")]
        figures = [h for h in PARTS if h.startswith("<img")]
        tables = [h for h in PARTS if h not in banners and h not in figures]
        figures = [h.replace("<img ", '<img style="width:100%;max-width:520px;height:auto" ', 1) for h in figures]
        if chart_beside_controls:
            chart.value = f'<div style="width:480px;max-width:100%">{"".join(figures)}</div>'
            out.value = "".join(banners) + "".join(tables)
            return
        out.value = ("".join(banners) +
                     '<div style="display:flex;flex-wrap:nowrap;gap:18px;align-items:flex-start">'
                     f'<div style="flex:0 0 auto">{"".join(tables)}</div>'
                     f'<div style="flex:1 1 280px;min-width:240px">{"".join(figures)}</div></div>')

    def load(values):
        for k, w in controls.items():
            w.unobserve(refresh, names="value")
            w.value = values(k)
            w.observe(refresh, names="value")
        refresh()

    b_opt = W.Button(description="Load the optimal plan", button_style="success", layout=W.Layout(width="200px"))
    b_zero = W.Button(description="Reset to 0", layout=W.Layout(width="120px"))
    b_opt.on_click(lambda _: load(lambda k: best[k]))
    b_zero.on_click(lambda _: load(lambda k: False if isinstance(controls[k], W.Checkbox) else 0.0))
    for w in controls.values():
        w.observe(refresh, names="value")
    refresh()
    top = W.HBox(cols + [chart]) if chart_beside_controls else W.HBox(cols)
    display(W.VBox([top, W.HBox([b_opt, b_zero]), out]))


_ANS = json.loads(base64.b64decode("eyJFMV9jb3N0IjogMi42NDcxLCAiRTFfbWlsayI6IDEuNDcwNiwgIkUyX2Nvc3QiOiAzNjcwLCAiRTJfUF9qYW4iOiA4NSwgIkUzX2Nvc3QiOiAzNDgwfQ=="))


def check(key, value, tol=0.01):
    """Compare your answer with the expected one (relative tolerance 1%)."""
    if value is None:
        print(f"⏳ {key}: not answered yet"); return
    exp = _ANS[key]
    ok = abs(float(value) - exp) <= tol * max(1.0, abs(exp))
    print(f"✅ {key}: correct" if ok else f"❌ {key}: not yet")


print("Ready. Run each explorer cell below, then move the sliders.")

---
## 1 · Diet: the cheapest hospital meal
A hospital needs a cost-effective meal that meets nutrition standards. Cost and nutrition content per unit:

| Food | Cost (\$) | Calories | Protein (g) | Vitamin C (mg) |
|---|---|---|---|---|
| Bread | 0.50 | 100 | 3 | 0 |
| Meat | 2.00 | 200 | 20 | 5 |
| Vegetables | 1.00 | 50 | 2 | 15 |

Daily requirements: at least 400 calories, 15 g of protein and 10 mg of vitamin C.

**Try this**
* Start from 0: which requirements fail? Add bread only: which requirement can bread never satisfy?
* Find a feasible meal, then make it cheaper step by step. Which requirements are at exactly 100%?
* Load the optimal plan: is the cheapest meal the one with the cheapest food?

In [ ]:
# @title 🍞 Diet explorer: choose the meal, watch cost and nutrition
def _explorer():
    FOODS = ["Bread", "Meat", "Vegetables"]
    PRICE = np.array([0.50, 2.00, 1.00])                   # $ per unit (fixed data)
    CONTENT = np.array([[100, 200, 50],                    # calories per unit
                        [3, 20, 2],                        # protein (g)
                        [0, 5, 15]])                       # vitamin C (mg)
    NEED = np.array([400, 15, 10])
    NUTRIENTS = ["Calories", "Protein (g)", "Vitamin C (mg)"]
    x_best, z_best = optimum(PRICE, A_ub=-CONTENT, b_ub=-NEED, bounds=[(0, None)] * 3)


    def diet_view(x1, x2, x3):
        x = np.array([x1, x2, x3])
        got = CONTENT @ x
        violated = constraints([(n, g, "≥", r) for n, g, r in zip(NUTRIENTS, got, NEED)])
        verdict(violated, PRICE @ x, z_best, "cost of the meal")
        table(pd.DataFrame({"Food": FOODS, "Units xⱼ": x, "Price ($/unit)": PRICE, "Cost ($)": PRICE * x}))
        fig, ax = plt.subplots(figsize=(5, 2.4))
        pct = 100 * got / NEED
        ax.barh(NUTRIENTS, pct, color=["#067429" if p >= 100 - 1e-6 else "#bd0a0a" for p in pct])
        ax.axvline(100, color="black", ls="--", lw=1)
        ax.set_xlabel("% of the daily requirement (dashed line: 100%)")
        ax.set_xlim(0, max(150, pct.max() * 1.1))
        show(fig)


    explorer({"Your decisions: units of each food": dict(x1=slider("x₁ bread", 0, 0, 8, 0.05),
                                                           x2=slider("x₂ meat", 0, 0, 8, 0.05),
                                                           x3=slider("x₃ vegetables", 0, 0, 8, 0.05))},
             diet_view, dict(zip(["x1", "x2", "x3"], x_best)))


_explorer()

### Formulation
**Variables:** $x_1, x_2, x_3$ = units of bread, meat and vegetables.

$$
\begin{aligned}
\min\ & z = 0.50x_1 + 2.00x_2 + 1.00x_3 \\
\text{s.t. } & 100x_1 + 200x_2 + 50x_3 \ge 400 && \text{(calories)} \\
& 3x_1 + 20x_2 + 2x_3 \ge 15 && \text{(protein)} \\
& 0x_1 + 5x_2 + 15x_3 \ge 10 && \text{(vitamin C)} \\
& x_1, x_2, x_3 \ge 0
\end{aligned}
$$

**General form:** $\min \sum_j c_j x_j$ s.t. $\sum_j a_{ij} x_j \ge r_i$ for every nutrient $i$, $x_j \ge 0$.

In [ ]:
import pulp

foods = ["Bread", "Meat", "Vegetables"]
cost = {"Bread": 0.50, "Meat": 2.00, "Vegetables": 1.00}          # $ per unit
content = {"Calories": {"Bread": 100, "Meat": 200, "Vegetables": 50},
           "Protein": {"Bread": 3, "Meat": 20, "Vegetables": 2},
           "VitaminC": {"Bread": 0, "Meat": 5, "Vegetables": 15}}
need = {"Calories": 400, "Protein": 15, "VitaminC": 10}           # daily minimum

model = pulp.LpProblem("diet", pulp.LpMinimize)
x = {j: model.add_variable(f"x_{j}", lowBound=0) for j in foods}     # units of each food
model += pulp.lpSum(cost[j] * x[j] for j in foods)                 # total cost
for i in need:
    model += pulp.lpSum(content[i][j] * x[j] for j in foods) >= need[i], i
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):.2f}")
for j in foods:
    print(f"  {j:<11} {x[j].value():6.2f} units")

---
## 2 · Inventory planning: produce now or later?
A company plans its production over three months and can carry inventory from one month to the next.

| Month | Demand | Production capacity | Production cost (\$/unit) |
|---|---|---|---|
| January | 80 | 100 | 8 |
| February | 95 | 90 | 9 |
| March | 110 | 100 | 11 |

Initial inventory: 15 units. Holding cost: \$2 per unit carried from one month to the next.
The inventory at the end of March must be 0.

The decisions are the productions $P_t$ and the ending inventories $I_t$. Once you choose the productions, the
**balance equation** $I_{t-1} + P_t = d_t + I_t$ fixes the inventories: the explorer computes them for you.

**Try this**
* Produce exactly the demand each month. Which month fails, and why?
* February's capacity (90) is below its demand (95). Where can the missing units come from, and what does it cost?
* Find a feasible plan, then load the optimal one: in which month is it worth producing in advance?

In [ ]:
# @title 📦 Inventory explorer: choose the production, watch stock and cost
def _explorer():
    MONTHS = ["January", "February", "March"]
    DEMAND = np.array([80, 95, 110]); CAP = np.array([100, 90, 100]); UNIT = np.array([8, 9, 11])
    HOLD, I0 = 2, 15                                       # fixed data
    A_eq = [[1, 0, 0, -1, 0, 0], [0, 1, 0, 1, -1, 0], [0, 0, 1, 0, 1, -1]]
    xb, z_best = optimum(np.r_[UNIT, [HOLD] * 3], A_eq=A_eq, b_eq=[DEMAND[0] - I0, DEMAND[1], DEMAND[2]],
                         bounds=[(0, c) for c in CAP] + [(0, None)] * 2 + [(0, 0)])


    def inventory_view(P1, P2, P3):
        P = np.array([P1, P2, P3])
        I = I0 + np.cumsum(P - DEMAND)                     # the balance equation gives the ending inventory
        rows = [(f"capacity {m}", p, "≤", c) for m, p, c in zip(MONTHS, P, CAP)]
        rows += [(f"inventory end of {m} (no shortage)", i, "≥", 0) for m, i in zip(MONTHS[:2], I[:2])]
        rows += [("inventory end of March (target)", I[2], "=", 0)]
        violated = constraints(rows)
        verdict(violated, UNIT @ P + HOLD * I.sum(), z_best, "total cost")
        table(pd.DataFrame({"Month": MONTHS, "Demand": DEMAND, "Production Pₜ": P,
                            "Ending inventory Iₜ": I, "Cost ($)": UNIT * P + HOLD * np.maximum(I, 0)}))
        fig, ax = plt.subplots(figsize=(5.4, 3.0))
        pos = np.arange(3)
        ax.bar(pos - 0.2, CAP, 0.4, color="none", edgecolor="#305bab", ls="--", label="capacity")
        ax.bar(pos - 0.2, P, 0.4, color="#659df2", label="production")
        ax.bar(pos + 0.2, I, 0.4, color=["#ffdc4a" if i >= 0 else "#bd0a0a" for i in I], label="ending inventory")
        ax.plot(pos, DEMAND, "ko-", label="demand"); ax.axhline(0, color="black", lw=0.8)
        ax.set_xticks(pos, MONTHS); ax.legend(fontsize=8, ncol=4, loc="upper left")
        ax.set_ylim(min(0, I.min()) * 1.2 - 5, 165)
        show(fig)


    explorer({"Your decisions: units produced": dict(P1=slider("P₁ January", 0, 0, 150, 5),
                                                      P2=slider("P₂ February", 0, 0, 150, 5),
                                                      P3=slider("P₃ March", 0, 0, 150, 5))},
             inventory_view, dict(zip(["P1", "P2", "P3"], xb[:3])))


_explorer()

### Formulation
**Variables:** $P_t$ = production in month $t$, $I_t$ = inventory at the end of month $t$.

$$
\begin{aligned}
\min\ & z = 8P_{Jan} + 9P_{Feb} + 11P_{Mar} + 2I_{Jan} + 2I_{Feb} + 2I_{Mar} \\
\text{s.t. } & 15 + P_{Jan} = 80 + I_{Jan} && \text{(balance, January)} \\
& I_{Jan} + P_{Feb} = 95 + I_{Feb} && \text{(balance, February)} \\
& I_{Feb} + P_{Mar} = 110 + I_{Mar} && \text{(balance, March)} \\
& P_{Jan} \le 100,\ P_{Feb} \le 90,\ P_{Mar} \le 100 && \text{(capacity)} \\
& I_{Mar} = 0 && \text{(final inventory)} \\
& P_t, I_t \ge 0
\end{aligned}
$$

**General form:** $\min \sum_t (c_t P_t + h_t I_t)$ s.t. $I_{t-1} + P_t = d_t + I_t$ and $P_t \le u_t$ for every
period $t$, $I_n = I^{\text{end}}$, $P_t, I_t \ge 0$ (with $I_0$ the initial inventory).

In [ ]:
import pulp

months = ["Jan", "Feb", "Mar"]
demand = {"Jan": 80, "Feb": 95, "Mar": 110}
capacity = {"Jan": 100, "Feb": 90, "Mar": 100}
prod_cost = {"Jan": 8, "Feb": 9, "Mar": 11}       # $ per unit produced
hold_cost, initial, final = 2, 15, 0              # $ per unit carried, units, units

model = pulp.LpProblem("inventory", pulp.LpMinimize)
P = {t: model.add_variable(f"P_{t}", lowBound=0) for t in months}   # production
I = {t: model.add_variable(f"I_{t}", lowBound=0) for t in months}   # ending inventory
model += pulp.lpSum(prod_cost[t] * P[t] + hold_cost * I[t] for t in months)
previous = initial                                   # inventory at the start of the month
for t in months:
    model += previous + P[t] == demand[t] + I[t], f"balance_{t}"
    model += P[t] <= capacity[t], f"capacity_{t}"
    previous = I[t]
model += I["Mar"] == final, "final_inventory"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
for t in months:
    print(f"  {t}: produce {P[t].value():5.0f}, ending inventory {I[t].value():4.0f}")

---
## 3 · Transportation: from factories to stores
A company with 3 factories ships its product to 4 retail stores. Shipping costs (\$ per unit):

| From \ To | Store A | Store B | Store C | Store D | Supply |
|---|---|---|---|---|---|
| Factory 1 | 8 | 6 | 10 | 9 | 100 |
| Factory 2 | 9 | 12 | 13 | 7 | 150 |
| Factory 3 | 14 | 9 | 16 | 5 | 120 |
| **Demand** | 80 | 90 | 110 | 90 | |

**Try this**
* Serve every store from its cheapest factory. Which constraint breaks?
* Build any feasible plan (all ✔), then lower its cost by moving 10 units at a time between routes.
* Load the optimal plan: how many routes does it use out of 12?

In [ ]:
# @title 🚚 Transportation explorer: choose the shipments, watch cost, supply and demand
def _explorer():
    FACTORIES, STORES = ["F1", "F2", "F3"], ["A", "B", "C", "D"]
    COST = np.array([[8, 6, 10, 9], [9, 12, 13, 7], [14, 9, 16, 5]])   # $ per unit (fixed data)
    SUPPLY, DEMAND = np.array([100, 150, 120]), np.array([80, 90, 110, 90])
    xb, z_best = optimum(COST.ravel(), A_ub=np.r_[np.kron(np.eye(3), np.ones(4)), -np.kron(np.ones(3), np.eye(4))],
                         b_ub=np.r_[SUPPLY, -DEMAND], bounds=[(0, None)] * 12)
    KEYS = [f"x_{f}_{s}" for f in FACTORIES for s in STORES]


    def transport_view(**x):
        X = np.array([x[k] for k in KEYS]).reshape(3, 4)
        rows = [(f"supply {f}", X[i].sum(), "≤", SUPPLY[i]) for i, f in enumerate(FACTORIES)]
        rows += [(f"demand store {s}", X[:, j].sum(), "≥", DEMAND[j]) for j, s in enumerate(STORES)]
        violated = constraints(rows)
        verdict(violated, (COST * X).sum(), z_best, "shipping cost")
        fig, ax = plt.subplots(figsize=(5.2, 3.0))
        yf, ys = [2, 0, -2], [3, 1, -1, -3]
        for a in range(3):
            for b in range(4):
                if X[a, b] > 1e-6:
                    ax.plot([0, 4], [yf[a], ys[b]], color="#305bab", lw=0.5 + 6 * X[a, b] / max(X.max(), 1), alpha=0.8)
                    t = 0.2 + 0.15 * b
                    ax.text(4 * t, yf[a] + t * (ys[b] - yf[a]), f"{X[a, b]:g}", ha="center", va="center", fontsize=8,
                            bbox=dict(facecolor="white", edgecolor="#305bab", boxstyle="round,pad=0.2"))
        for k, f in enumerate(FACTORIES):
            ax.scatter(0, yf[k], s=600, color="#c6dcff", zorder=3); ax.text(0, yf[k], f, ha="center", va="center", zorder=4)
        for k, s in enumerate(STORES):
            ax.scatter(4, ys[k], s=600, color="#adf0c7", zorder=3); ax.text(4, ys[k], s, ha="center", va="center", zorder=4)
        ax.set_xlim(-0.6, 4.6); ax.axis("off"); ax.set_title("your shipments (units)", fontsize=10)
        show(fig)


    groups = {f"Your decisions: shipments from {f}": {f"x_{f}_{s}": slider(f"{f} → {s} (${COST[i, j]}/u)", 0, 0, 150, 5)
                                                       for j, s in enumerate(STORES)}
              for i, f in enumerate(FACTORIES)}
    explorer(groups, transport_view, dict(zip(KEYS, xb)))


_explorer()

### Formulation
**Variables:** $x_{ij}$ = units shipped from factory $i \in \{1, 2, 3\}$ to store $j \in \{A, B, C, D\}$
(12 variables).

$$
\begin{aligned}
\min\ z = &\ 8x_{1A} + 6x_{1B} + 10x_{1C} + 9x_{1D} + 9x_{2A} + 12x_{2B} + 13x_{2C} + 7x_{2D} \\
&+ 14x_{3A} + 9x_{3B} + 16x_{3C} + 5x_{3D} \\
\text{s.t. } & x_{1A} + x_{1B} + x_{1C} + x_{1D} \le 100 && \text{(supply F1)} \\
& x_{2A} + x_{2B} + x_{2C} + x_{2D} \le 150 && \text{(supply F2)} \\
& x_{3A} + x_{3B} + x_{3C} + x_{3D} \le 120 && \text{(supply F3)} \\
& x_{1A} + x_{2A} + x_{3A} \ge 80,\quad x_{1B} + x_{2B} + x_{3B} \ge 90 && \text{(demand A, B)} \\
& x_{1C} + x_{2C} + x_{3C} \ge 110,\quad x_{1D} + x_{2D} + x_{3D} \ge 90 && \text{(demand C, D)} \\
& x_{ij} \ge 0
\end{aligned}
$$

**General form:** $\min \sum_i \sum_j c_{ij} x_{ij}$ s.t. $\sum_j x_{ij} \le S_i$ for every source $i$,
$\sum_i x_{ij} \ge D_j$ for every destination $j$, $x_{ij} \ge 0$. Feasible only if $\sum_i S_i \ge \sum_j D_j$.

In [ ]:
import pulp
import pandas as pd

factories, stores = ["F1", "F2", "F3"], ["A", "B", "C", "D"]
supply = {"F1": 100, "F2": 150, "F3": 120}
demand = {"A": 80, "B": 90, "C": 110, "D": 90}
cost_table = [[8, 6, 10, 9],                      # $ per unit, one row per factory
              [9, 12, 13, 7],
              [14, 9, 16, 5]]
cost = {(f, s): cost_table[i][j] for i, f in enumerate(factories) for j, s in enumerate(stores)}

model = pulp.LpProblem("transportation", pulp.LpMinimize)
x = {(f, s): model.add_variable(f"x_{f}_{s}", lowBound=0) for (f, s) in cost}   # one variable per route
model += pulp.lpSum(cost[r] * x[r] for r in cost)
for f in factories:
    model += pulp.lpSum(x[f, s] for s in stores) <= supply[f], f"supply_{f}"
for s in stores:
    model += pulp.lpSum(x[f, s] for f in factories) >= demand[s], f"demand_{s}"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
pd.DataFrame([[x[f, s].value() for s in stores] for f in factories], index=factories, columns=stores)

---
## 4 · Shortest path: the fastest way to campus
You drive from home $O$ to campus $D$ through a small network of **one-way** roads, with these travel times:

| Road | O→A | O→B | B→A | A→C | A→E | B→E | C→E | C→D | E→D |
|---|---|---|---|---|---|---|---|---|---|
| Time (min) | 9 | 4 | 2 | 5 | 6 | 12 | 2 | 6 | 3 |

**Try this**
* Tick roads one by one to build a route from O to D. Watch the *flow out − flow in* rows: why must they be
  +1 at O, −1 at D and 0 elsewhere?
* Tick a road that does not connect to your route. Which node balance breaks?
* Find the fastest route yourself, then load the optimal one.

In [ ]:
# @title 🗺️ Shortest path explorer: choose the roads, watch the flow balance and the time
def _explorer():
    ROADS = [("O", "A"), ("O", "B"), ("B", "A"), ("A", "C"), ("A", "E"), ("B", "E"), ("C", "E"), ("C", "D"), ("E", "D")]
    TIME = np.array([9, 4, 2, 5, 6, 12, 2, 6, 3])           # minutes per one-way road (fixed data)
    SPOTS = {"O": (0, 0), "A": (1, 1), "B": (1, -1), "C": (2, 1), "E": (2, -1), "D": (3, 0)}
    NEED = {k: 1 if k == "O" else -1 if k == "D" else 0 for k in SPOTS}
    A_eq = [[(1 if a == k else 0) - (1 if b == k else 0) for a, b in ROADS] for k in SPOTS]
    xb, z_best = optimum(TIME, A_eq=A_eq, b_eq=list(NEED.values()), bounds=[(0, None)] * len(ROADS))
    KEYS = [f"x_{a}{b}" for a, b in ROADS]


    def path_view(**x):
        v = np.array([float(x[k]) for k in KEYS])
        out_in = np.array(A_eq) @ v
        violated = constraints([(f"node {k}: flow out − flow in", oi, "=", NEED[k]) for k, oi in zip(SPOTS, out_in)])
        verdict(violated, TIME @ v, z_best, "travel time", unit="min")
        fig, ax = plt.subplots(figsize=(5.4, 2.8))
        for (a, b), t, used in zip(ROADS, TIME, v):
            (x0, y0), (x1, y1) = SPOTS[a], SPOTS[b]
            color = "#067429" if used else "#9aa5b1"
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=4 if used else 1.2, shrinkA=14, shrinkB=14))
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, f"{t} min", ha="center", va="center", fontsize=9, color=color,
                    bbox=dict(facecolor="white", edgecolor=color, boxstyle="round,pad=0.2"))
        for k, (px, py) in SPOTS.items():
            ax.scatter(px, py, s=700, color="#adf0c7" if k in "OD" else "#c6dcff", zorder=3)
            ax.text(px, py, k, ha="center", va="center", zorder=4, fontweight="bold")
        ax.set_xlim(-0.4, 3.4); ax.set_ylim(-1.5, 1.5); ax.axis("off")
        ax.set_title("one-way roads (green: the roads you use)", fontsize=10)
        show(fig)


    boxes = {k: W.Checkbox(value=False, description=f"use {a}→{b} ({t} min)", indent=False)
             for k, (a, b), t in zip(KEYS, ROADS, TIME)}
    half = (len(KEYS) + 1) // 2                              # two columns of check boxes
    explorer({"Your decisions: xᵢⱼ = 1 if you drive on road (i, j)": {k: boxes[k] for k in KEYS[:half]},
              "&nbsp;": {k: boxes[k] for k in KEYS[half:]}}, path_view,
             {k: bool(round(v)) for k, v in zip(KEYS, xb)}, chart_beside_controls=True)


_explorer()

### Formulation
**Variables:** $x_{ij} = 1$ if you drive on road $(i, j)$, 0 otherwise (the LP only needs $x_{ij} \ge 0$).

$$
\begin{aligned}
\min\ & z = 9x_{OA} + 4x_{OB} + 2x_{BA} + 5x_{AC} + 6x_{AE} + 12x_{BE} + 2x_{CE} + 6x_{CD} + 3x_{ED} \\
\text{s.t. } & x_{OA} + x_{OB} = 1 && \text{(origin } O\text{: out − in = 1)} \\
& x_{AC} + x_{AE} - x_{OA} - x_{BA} = 0 && (A) \\
& x_{BA} + x_{BE} - x_{OB} = 0 && (B) \\
& x_{CE} + x_{CD} - x_{AC} = 0 && (C) \\
& x_{ED} - x_{AE} - x_{BE} - x_{CE} = 0 && (E) \\
& -x_{CD} - x_{ED} = -1 && \text{(destination } D\text{: out − in = −1)} \\
& x_{ij} \ge 0
\end{aligned}
$$

**General form:** $\min \sum_{(i,j)} t_{ij} x_{ij}$ s.t.
$\sum_{j:(k,j)} x_{kj} - \sum_{i:(i,k)} x_{ik} = b_k$ for every node $k$, with $b_O = 1$, $b_D = -1$ and $b_k = 0$
otherwise, $x_{ij} \ge 0$. The model never requires $x_{ij} \in \{0, 1\}$, yet the optimal solution is 0/1.

In [ ]:
import pulp

nodes = ["O", "A", "B", "C", "E", "D"]
time = {("O", "A"): 9, ("O", "B"): 4, ("B", "A"): 2, ("A", "C"): 5, ("A", "E"): 6,
        ("B", "E"): 12, ("C", "E"): 2, ("C", "D"): 6, ("E", "D"): 3}     # minutes per one-way road
origin, destination = "O", "D"

model = pulp.LpProblem("shortest_path", pulp.LpMinimize)
x = {(i, j): model.add_variable(f"x_{i}{j}", lowBound=0) for (i, j) in time}   # 1 if the road is used
model += pulp.lpSum(time[r] * x[r] for r in time)
for k in nodes:
    flow_out = pulp.lpSum(x[i, j] for (i, j) in time if i == k)
    flow_in = pulp.lpSum(x[i, j] for (i, j) in time if j == k)
    b = 1 if k == origin else -1 if k == destination else 0
    model += flow_out - flow_in == b, f"flow_{k}"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· travel time = {pulp.value(model.objective):g} min")
print("  roads used:", ", ".join(f"{i}→{j}" for (i, j) in time if x[i, j].value() > 0.5))

---
## 5 · Maximum flow: how much water can reach the plant?
A water company wants to know the maximum flow from a reservoir (source $S$) to a treatment plant (sink $T$)
through a network of pipes. Capacities (units/hour):

| Pipe | S→A | S→B | A→C | A→D | B→C | B→D | C→T | D→T |
|---|---|---|---|---|---|---|---|---|
| Capacity | 5 | 8 | 5 | 3 | 4 | 7 | 8 | 6 |

**Try this**
* Open S→A to 5 only. Which conservation rows fail? Push the water through until every node balances.
* Increase the flow step by step. Which pipes are full (flow = capacity) when you cannot go further?
* Load the optimal plan: can the water leaving S ever exceed 5 + 8?

In [ ]:
# @title 💧 Maximum flow explorer: choose the flows, watch capacities and conservation
def _explorer():
    ARCS = [("S", "A"), ("S", "B"), ("A", "C"), ("A", "D"), ("B", "C"), ("B", "D"), ("C", "T"), ("D", "T")]
    CAP = np.array([5, 8, 5, 3, 4, 7, 8, 6])                 # units/hour (fixed data)
    POS = {"S": (0, 0), "A": (1, 1), "B": (1, -1), "C": (2, 1), "D": (2, -1), "T": (3, 0)}
    INNER = ["A", "B", "C", "D"]
    A_eq = [[(1 if a == k else 0) - (1 if b == k else 0) for a, b in ARCS] for k in INNER]
    xb, z_best = optimum([1 if a == "S" else 0 for a, _ in ARCS], A_eq=A_eq, b_eq=[0] * 4,
                         bounds=[(0, c) for c in CAP], maximize=True)
    KEYS = [f"x_{a}{b}" for a, b in ARCS]


    def flow_view(**x):
        f = np.array([x[k] for k in KEYS])
        rows = [(f"capacity {a}→{b}", v, "≤", c) for (a, b), v, c in zip(ARCS, f, CAP)]
        rows += [(f"node {k}: flow out − flow in", oi, "=", 0) for k, oi in zip(INNER, np.array(A_eq) @ f)]
        violated = constraints(rows)
        verdict(violated, sum(v for (a, _), v in zip(ARCS, f) if a == "S"), z_best, "flow leaving S",
                maximize=True, unit="units/hour")
        fig, ax = plt.subplots(figsize=(5.4, 2.8))
        for (a, b), v, c in zip(ARCS, f, CAP):
            (x0, y0), (x1, y1) = POS[a], POS[b]
            color = "#bd0a0a" if v > c else "#305bab"
            ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=1 + 5 * v / max(CAP.max(), 1), shrinkA=14, shrinkB=14))
            t = 0.35 if x0 != x1 and y0 != y1 and a != "S" and b != "T" else 0.5
            ax.text(x0 + t * (x1 - x0), y0 + t * (y1 - y0), f"{v:g}/{c}", ha="center", va="center", fontsize=9,
                    color=color, bbox=dict(facecolor="white", edgecolor=color, boxstyle="round,pad=0.2"))
        for k, (px, py) in POS.items():
            ax.scatter(px, py, s=700, color="#adf0c7" if k in "ST" else "#c6dcff", zorder=3)
            ax.text(px, py, k, ha="center", va="center", zorder=4, fontweight="bold")
        ax.set_xlim(-0.4, 3.4); ax.set_ylim(-1.5, 1.5); ax.axis("off")
        ax.set_title("your flow / capacity on each pipe (units/hour)", fontsize=10)
        show(fig)


    pipes = {k: slider(f"{a}→{b} (cap. {c})", 0, 0, 10, 1) for k, (a, b), c in zip(KEYS, ARCS, CAP)}
    for w in pipes.values():                                 # narrower sliders: two columns + the map side by side
        w.layout.width, w.style.description_width = "290px", "105px"
    half = len(KEYS) // 2
    explorer({"Your decisions: flow on each pipe": {k: pipes[k] for k in KEYS[:half]},
              "&nbsp;": {k: pipes[k] for k in KEYS[half:]}}, flow_view, dict(zip(KEYS, xb)), chart_beside_controls=True)


_explorer()

### Formulation
**Variables:** $x_{ij}$ = flow on pipe $(i, j)$ (8 variables).

$$
\begin{aligned}
\max\ & z = x_{SA} + x_{SB} && \text{(flow leaving the source)} \\
\text{s.t. } & x_{SA} \le 5,\ x_{SB} \le 8,\ x_{AC} \le 5,\ x_{AD} \le 3 && \text{(capacity)} \\
& x_{BC} \le 4,\ x_{BD} \le 7,\ x_{CT} \le 8,\ x_{DT} \le 6 && \text{(capacity)} \\
& x_{AC} + x_{AD} - x_{SA} = 0,\quad x_{BC} + x_{BD} - x_{SB} = 0 && \text{(conservation at } A, B) \\
& x_{CT} - x_{AC} - x_{BC} = 0,\quad x_{DT} - x_{AD} - x_{BD} = 0 && \text{(conservation at } C, D) \\
& x_{ij} \ge 0
\end{aligned}
$$

**General form:** $\max \sum_{j:(s,j)} x_{sj}$ s.t. $x_{ij} \le u_{ij}$ for every arc,
$\sum_{j:(i,j)} x_{ij} - \sum_{k:(k,i)} x_{ki} = 0$ for every node $i \ne s, t$, $x_{ij} \ge 0$.

In [ ]:
import pulp

capacity = {("S", "A"): 5, ("S", "B"): 8, ("A", "C"): 5, ("A", "D"): 3,
            ("B", "C"): 4, ("B", "D"): 7, ("C", "T"): 8, ("D", "T"): 6}   # units/hour
source, sink = "S", "T"
junctions = ["A", "B", "C", "D"]

model = pulp.LpProblem("max_flow", pulp.LpMaximize)
x = {(i, j): model.add_variable(f"x_{i}{j}", lowBound=0) for (i, j) in capacity}   # flow on each pipe
model += pulp.lpSum(x[i, j] for (i, j) in capacity if i == source)
for (i, j) in capacity:
    model += x[i, j] <= capacity[i, j], f"capacity_{i}{j}"
for k in junctions:
    flow_out = pulp.lpSum(x[i, j] for (i, j) in capacity if i == k)
    flow_in = pulp.lpSum(x[i, j] for (i, j) in capacity if j == k)
    model += flow_out - flow_in == 0, f"conservation_{k}"
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· maximum flow = {pulp.value(model.objective):g} units/hour")
for (i, j), u in capacity.items():
    print(f"  {i}→{j}: {x[i, j].value():g} / {u}")

---
## What did you learn?
* The **data** (prices, capacities, demands) describe the situation and never change while you decide; the
  **decision variables** are what you control.
* A plan is **feasible** only if it satisfies every constraint; among the feasible plans, the solver finds the one
  with the best objective.
* At the optimum, a few constraints are **binding** (used exactly at their limit): they are what stops any
  further improvement.
* Shortest path and maximum flow are both **network** models: one flow-balance constraint per node.

---
## Your turn: modify a model and solve it
In each exercise, start from the PuLP code of the problem above, change what the situation requires, solve
it, and run the ✅ check. Write the model first (on paper): **what changes, the data or the structure?**

### Exercise 1 · Diet: a new food and a palatability rule
The hospital can now also serve **milk**: \$0.80 per unit, with 120 calories, 8 g of protein and 2 mg of vitamin C
per unit. Patients complain about bread: **at most 2 units of bread** per meal.
What is the new cheapest meal? Does meat stay in the meal?

In [ ]:
import pulp

foods = ["Bread", "Meat", "Vegetables"]                          # ✍️ add "Milk"
cost = {"Bread": 0.50, "Meat": 2.00, "Vegetables": 1.00}          # ✍️ add the cost of milk
content = {"Calories": {"Bread": 100, "Meat": 200, "Vegetables": 50},   # ✍️ add milk to each nutrient
           "Protein": {"Bread": 3, "Meat": 20, "Vegetables": 2},
           "VitaminC": {"Bread": 0, "Meat": 5, "Vegetables": 15}}
need = {"Calories": 400, "Protein": 15, "VitaminC": 10}

model = pulp.LpProblem("diet_milk", pulp.LpMinimize)
x = {j: model.add_variable(f"x_{j}", lowBound=0) for j in foods}
model += pulp.lpSum(cost[j] * x[j] for j in foods)
for i in need:
    model += pulp.lpSum(content[i][j] * x[j] for j in foods) >= need[i], i
# ✍️ add the palatability rule: at most 2 units of bread
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):.4f}")
for j in foods:
    print(f"  {j:<11} {x[j].value():6.3f} units")
E1_cost = None          # ✍️ optimal cost of the meal ($)
E1_milk = None          # ✍️ units of milk in the optimal meal
check("E1_cost", E1_cost); check("E1_milk", E1_milk)

### Exercise 2 · Inventory: plan one more month
The plan now covers **April** too: demand 105 units, capacity 100, production cost \$10 per unit (holding cost
still \$2). The inventory must be 0 at the end of **April** (no longer March).
What is the new total cost, and how many units are produced in January? Why more than before?

In [ ]:
import pulp

months = ["Jan", "Feb", "Mar"]                     # ✍️ add "Apr"
demand = {"Jan": 80, "Feb": 95, "Mar": 110}        # ✍️ April: 105
capacity = {"Jan": 100, "Feb": 90, "Mar": 100}     # ✍️ April: 100
prod_cost = {"Jan": 8, "Feb": 9, "Mar": 11}        # ✍️ April: 10
hold_cost, initial, final = 2, 15, 0

model = pulp.LpProblem("inventory_april", pulp.LpMinimize)
P = {t: model.add_variable(f"P_{t}", lowBound=0) for t in months}
I = {t: model.add_variable(f"I_{t}", lowBound=0) for t in months}
model += pulp.lpSum(prod_cost[t] * P[t] + hold_cost * I[t] for t in months)
previous = initial
for t in months:
    model += previous + P[t] == demand[t] + I[t], f"balance_{t}"
    model += P[t] <= capacity[t], f"capacity_{t}"
    previous = I[t]
model += I["Mar"] == final, "final_inventory"      # ✍️ the final month is now April
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
for t in months:
    print(f"  {t}: produce {P[t].value():5.0f}, ending inventory {I[t].value():4.0f}")
E2_cost = None          # ✍️ optimal total cost ($)
E2_P_jan = None         # ✍️ units produced in January
check("E2_cost", E2_cost); check("E2_P_jan", E2_P_jan)

### Exercise 3 · Transportation: a new configuration
Factory 1 expands to **120 units**, Store B now needs **110 units**, and the road **F3 → D is closed** (no
shipment possible on that route). What is the new cheapest shipping plan?

*Hint:* a closed route can be removed from the model (no variable), or kept with the constraint $x_{F3,D} = 0$.

In [ ]:
import pulp
import pandas as pd

factories, stores = ["F1", "F2", "F3"], ["A", "B", "C", "D"]
supply = {"F1": 100, "F2": 150, "F3": 120}        # ✍️ Factory 1 expands
demand = {"A": 80, "B": 90, "C": 110, "D": 90}     # ✍️ Store B needs more
cost_table = [[8, 6, 10, 9],
              [9, 12, 13, 7],
              [14, 9, 16, 5]]
cost = {(f, s): cost_table[i][j] for i, f in enumerate(factories) for j, s in enumerate(stores)}

model = pulp.LpProblem("transportation_new", pulp.LpMinimize)
x = {(f, s): model.add_variable(f"x_{f}_{s}", lowBound=0) for (f, s) in cost}
model += pulp.lpSum(cost[r] * x[r] for r in cost)
for f in factories:
    model += pulp.lpSum(x[f, s] for s in stores) <= supply[f], f"supply_{f}"
for s in stores:
    model += pulp.lpSum(x[f, s] for f in factories) >= demand[s], f"demand_{s}"
# ✍️ close the road F3 → D
result = model.solve(pulp.COIN_CMD(msg=False))

print(result.status_str, f"· cost = ${pulp.value(model.objective):,.0f}")
display(pd.DataFrame([[x[f, s].value() for s in stores] for f in factories], index=factories, columns=stores))
E3_cost = None          # ✍️ optimal shipping cost ($)
check("E3_cost", E3_cost)